In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import datetime
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_validate
from tqdm import tqdm

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/predicting-stock-trends-rise-or-fall/sample_submission.csv
/kaggle/input/predicting-stock-trends-rise-or-fall/train.csv
/kaggle/input/predicting-stock-trends-rise-or-fall/test.csv


# Load Data

In [2]:
data_path = dirname

In [3]:
df_train = pd.read_csv(f"{data_path}/train.csv", parse_dates=["Date"])
df_test = pd.read_csv(f"{data_path}/test.csv", parse_dates=["Date"])
df_submission = pd.read_csv(f"{data_path}/sample_submission.csv")

In [4]:
TRAIN_FROM = "1962-01-01"

In [5]:
df_train = df_train[df_train["Date"] >= TRAIN_FROM]

In [6]:
df_train.head()

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,ticker_1,1962-01-02,0.000000,0.265828,0.261788,0.261788,25600.0,0.0,0.0
1,ticker_20,1962-01-02,0.000000,0.417455,0.412380,0.414917,84139.0,0.0,0.0
2,ticker_19,1962-01-02,0.000000,0.101537,0.100789,0.100789,902400.0,0.0,0.0
3,ticker_18,1962-01-02,0.000000,0.903030,0.881959,0.881959,51552.0,0.0,0.0
4,ticker_17,1962-01-02,0.130512,0.131783,0.129241,0.130512,163200.0,0.0,0.0


In [7]:
df_train.tail()

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
21033517,ticker_2098,2024-09-23,15.230000,15.230000,14.690000,14.780000,63400.0,0.0,0.0
21033518,ticker_3748,2024-09-23,24.700001,24.768400,24.510000,24.570000,31208.0,0.0,0.0
21033519,ticker_2615,2024-09-23,11.090000,11.150000,10.960000,11.090000,1014300.0,0.0,0.0
21033520,ticker_4765,2024-09-23,25.200001,25.202999,25.129999,25.134001,4500.0,0.0,0.0
21033521,ticker_4658,2024-09-23,0.281000,0.290000,0.264000,0.270000,272200.0,0.0,0.0


In [8]:
df_train.describe()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
count,21033522,2.103352e+07,2.103352e+07,2.103352e+07,2.103352e+07,2.103352e+07,2.103352e+07,21033522.0
mean,2010-07-13 05:38:49.608935424,2.285003e+01,2.335000e+01,2.265493e+01,2.300253e+01,1.174856e+06,2.012335e-03,inf
min,1962-01-02 00:00:00,0.000000e+00,7.606973e-02,0.000000e+00,7.425315e-02,0.000000e+00,0.000000e+00,0.0
25%,2003-06-10 00:00:00,4.680000e+00,4.900000e+00,4.727687e+00,4.810000e+00,2.550000e+04,0.000000e+00,0.0
50%,2013-05-06 00:00:00,1.162689e+01,1.189265e+01,1.151079e+01,1.170101e+01,1.452000e+05,0.000000e+00,0.0
75%,2020-02-27 00:00:00,2.584961e+01,2.627552e+01,2.554281e+01,2.591151e+01,6.881000e+05,0.000000e+00,0.0
max,2024-09-23 00:00:00,8.319200e+02,1.010080e+03,4.320597e+02,4.356200e+02,3.565254e+09,1.594000e+02,inf
std,NaN,3.409086e+01,3.457287e+01,3.360952e+01,3.408867e+01,6.574807e+06,8.486632e-02,NaN


In [9]:
df_train[df_train["Stock Splits"] != 0.0]

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
4632,ticker_18,1962-11-19,0.000000,0.716712,0.703970,0.713527,70243.0,0.06484,1.020000
5432,ticker_15,1963-01-15,0.000000,0.424637,0.424637,0.424637,0.0,0.00000,2.000000
12513,ticker_16,1964-05-18,1.620724,1.620724,1.595161,1.612203,583668.0,0.00000,1.250000
12972,ticker_5,1964-06-18,0.000000,0.979726,0.958051,0.962386,48000.0,0.00000,3.000000
13204,ticker_17,1964-07-06,0.249560,0.249560,0.242300,0.249560,15600.0,0.00000,2.000000
...,...,...,...,...,...,...,...,...,...
17561129,ticker_2717,2021-11-18,130.000000,133.250000,129.000000,132.550003,2126500.0,0.00000,4.000000
17640246,ticker_3586,2021-12-14,1.520000,1.670000,1.520000,1.530000,960200.0,0.00000,0.500000
17641566,ticker_2487,2021-12-14,21.048168,21.048168,20.055798,20.819923,25800.0,0.00000,1.000000
17767051,ticker_1202,2022-01-24,73.923454,75.648828,71.027989,75.172859,1083300.0,0.00000,1.000000


# Feature Engineering

In [10]:
# Create Label: Get the closing price after 30 business days for each stock, 1 if up, 0 if down
df_train = df_train.sort_values(["Ticker", "Date"])
df_train["future_close"] = df_train.groupby("Ticker")["Close"].shift(-30)
df_train["target"] = (df_train["future_close"] > df_train["Close"]).astype(int)

In [11]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["Ticker", "Date"])
    # Daily return
    df["ret_1"] = df.groupby("Ticker")["Close"].pct_change(1)
    # Moving average (5, 10, 20 days)
    for w in [5, 10, 20]:
        df[f"ma_{w}"] = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(w).mean())
    # volatility (5,10,20日)
    for w in [5,10,20]:
        df[f'vol_{w}'] = df.groupby('Ticker')['ret_1'].transform(lambda x: x.rolling(w).std())
    return df

In [12]:
df_train = create_features(df_train)

In [13]:
df_train.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,future_close,target,ret_1,ma_5,ma_10,ma_20,vol_5,vol_10,vol_20
0,ticker_1,1962-01-02,0.0,0.265828,0.261788,0.261788,25600.0,0.0,0.0,0.254992,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,ticker_1,1962-01-03,0.0,0.263808,0.261788,0.261788,28800.0,0.0,0.0,0.258663,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
49,ticker_1,1962-01-04,0.0,0.263404,0.260980,0.260980,21600.0,0.0,0.0,0.261927,1,-0.003086,NaN,NaN,NaN,NaN,NaN,NaN
76,ticker_1,1962-01-05,0.0,0.259364,0.255324,0.255324,46400.0,0.0,0.0,0.258256,1,-0.021673,NaN,NaN,NaN,NaN,NaN,NaN
97,ticker_1,1962-01-08,0.0,0.259364,0.255728,0.256536,29600.0,0.0,0.0,0.259071,1,0.004747,0.259283,NaN,NaN,NaN,NaN,NaN


In [14]:
df_train.tail()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,future_close,target,ret_1,ma_5,ma_10,ma_20,vol_5,vol_10,vol_20
21010467,ticker_999,2024-09-17,87.059998,88.120003,87.059998,87.709999,997900.0,0.0,0.0,NaN,0,0.009553,85.925998,84.388999,84.756499,0.005079,0.007838,0.017540
21014125,ticker_999,2024-09-18,87.720001,87.959999,86.680000,86.870003,757100.0,0.0,0.0,NaN,0,-0.009577,86.429999,84.770999,84.766999,0.009428,0.009242,0.017685
21019828,ticker_999,2024-09-19,88.220001,89.089996,87.680000,88.800003,885900.0,0.0,0.0,NaN,0,0.022217,87.164000,85.353999,84.873999,0.011999,0.010525,0.018352
21026462,ticker_999,2024-09-20,89.019997,90.070000,88.610001,89.910004,1665400.0,0.0,0.0,NaN,0,0.012500,88.034001,86.112000,85.031000,0.011921,0.009282,0.018520
21032385,ticker_999,2024-09-23,90.070000,90.239998,89.480003,89.989998,569600.0,0.0,0.0,NaN,0,0.000890,88.656001,86.809000,85.116500,0.012046,0.009623,0.018159


In [15]:
# Columns used for training
feature_cols = ['ret_1'] + [f'ma_{w}' for w in [5,10,20]] + [f'vol_{w}' for w in [5,10,20]]

In [16]:
feature_cols

['ret_1', 'ma_5', 'ma_10', 'ma_20', 'vol_5', 'vol_10', 'vol_20']

In [17]:
# Extract data for training (only rows with complete labels and features)
train_model = df_train.dropna(subset=['target', 'future_close'] + feature_cols)
X = train_model[feature_cols]
y = train_model['target']

In [18]:
train_model.tail()

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,future_close,target,ret_1,ma_5,ma_10,ma_20,vol_5,vol_10,vol_20
20860895,ticker_999,2024-08-05,77.559998,79.220001,76.980003,79.099998,777200.0,0.0,0.0,87.709999,1,-0.020191,81.860001,84.455001,85.483001,0.027185,0.021328,0.016789
20864330,ticker_999,2024-08-06,79.300003,81.349998,78.949997,80.949997,1572600.0,0.0,0.0,86.870003,1,0.023388,81.550000,83.804001,85.300500,0.024080,0.023684,0.017745
20869335,ticker_999,2024-08-07,82.349998,82.470001,79.820000,80.209999,741700.0,0.0,0.0,88.800003,1,-0.009141,80.725999,83.200001,85.047501,0.019313,0.023590,0.017650
20878678,ticker_999,2024-08-08,80.669998,81.389999,80.610001,81.309998,1008600.0,0.0,0.0,89.910004,1,0.013714,80.459999,82.633000,84.828500,0.020697,0.024026,0.017962
20880467,ticker_999,2024-08-09,81.379997,81.599998,81.019997,81.559998,668700.0,0.0,0.0,89.989998,1,0.003075,80.625998,82.056000,84.574500,0.017409,0.023982,0.017732


In [19]:
# Split into training/validation data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [20]:
# Train Model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='logloss'
)

In [21]:
model.fit(
    X_train, y_train,
    eval_set=[(X_train,y_train),(X_val,y_val)],
    verbose=True
)

[0]	validation_0-logloss:0.68940	validation_1-logloss:0.68941
[1]	validation_0-logloss:0.68892	validation_1-logloss:0.68892
[2]	validation_0-logloss:0.68851	validation_1-logloss:0.68852
[3]	validation_0-logloss:0.68818	validation_1-logloss:0.68819
[4]	validation_0-logloss:0.68789	validation_1-logloss:0.68790
[5]	validation_0-logloss:0.68765	validation_1-logloss:0.68766
[6]	validation_0-logloss:0.68742	validation_1-logloss:0.68744
[7]	validation_0-logloss:0.68726	validation_1-logloss:0.68727
[8]	validation_0-logloss:0.68709	validation_1-logloss:0.68711
[9]	validation_0-logloss:0.68696	validation_1-logloss:0.68698
[10]	validation_0-logloss:0.68684	validation_1-logloss:0.68687
[11]	validation_0-logloss:0.68675	validation_1-logloss:0.68677
[12]	validation_0-logloss:0.68667	validation_1-logloss:0.68670
[13]	validation_0-logloss:0.68661	validation_1-logloss:0.68664
[14]	validation_0-logloss:0.68656	validation_1-logloss:0.68659
[15]	validation_0-logloss:0.68650	validation_1-logloss:0.68653
[1

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [22]:
# Create features & predictions on test data
# Combine train data and test brand list and generate features with the same function
df_all = pd.concat([
    df_train[['Ticker','Date','Close']],
    df_test.rename(columns={'ID':'Ticker'})[['Ticker','Date']]
], ignore_index=True, sort=False)
df_all = create_features(df_all)

/tmp/ipykernel_31/4260301780.py:4: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["ret_1"] = df.groupby("Ticker")["Close"].pct_change(1)


In [23]:
df_all.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Ticker,Date,Close,ret_1,ma_5,ma_10,ma_20,vol_5,vol_10,vol_20
0,ticker_1,1962-01-02,0.261788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ticker_1,1962-01-03,0.261788,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,ticker_1,1962-01-04,0.260980,-0.003086,NaN,NaN,NaN,NaN,NaN,NaN
3,ticker_1,1962-01-05,0.255324,-0.021673,NaN,NaN,NaN,NaN,NaN,NaN
4,ticker_1,1962-01-08,0.256536,0.004747,0.259283,NaN,NaN,NaN,NaN,NaN


In [24]:
# Extract only features for the forecast target date (2024-11-04)
test_feats = df_all[df_all['Date']==pd.to_datetime('2024-11-04')]
X_test = test_feats.set_index('Ticker')[feature_cols]

In [25]:
# Predict
preds = model.predict(X_test)

In [26]:
# Make Submission File
submission = df_test.copy()
submission['Pred'] = submission['ID'].map(dict(zip(X_test.index, preds)))

In [27]:
submission.describe()

,Date,Pred
count,5000,5000.000000
mean,2024-11-04 00:00:00,0.069400
min,2024-11-04 00:00:00,0.000000
25%,2024-11-04 00:00:00,0.000000
50%,2024-11-04 00:00:00,0.000000
75%,2024-11-04 00:00:00,0.000000
max,2024-11-04 00:00:00,1.000000
std,NaN,0.254159


In [28]:
submission.head()

,ID,Date,Pred
0,ticker_1,2024-11-04,0
1,ticker_10,2024-11-04,0
2,ticker_100,2024-11-04,0
3,ticker_1000,2024-11-04,0
4,ticker_1001,2024-11-04,0


In [29]:
submission.drop(["Date"], axis=1).to_csv('submission.csv', index=False)

# Cross Validation

In [30]:
# TimeSeriesSplit with 5 divisions
tscv = TimeSeriesSplit(n_splits=5)

In [31]:
# When multiple evaluation indicators are calculated at once
cv_results = cross_validate(
    model,
    X,
    y,
    cv=tscv,
    scoring=['accuracy','roc_auc'],
    return_train_score=True
)

In [32]:
print("Accuracy for each division:", cv_results['test_accuracy'])
print("Average accuracy:", cv_results['test_accuracy'].mean())
print("AUC for each division:", cv_results['test_roc_auc'])
print("Average AUC:", cv_results['test_roc_auc'].mean())

Accuracy for each division: [0.55585031 0.54630505 0.53341474 0.54654986 0.55395188]
Average accuracy: 0.5472143698468787
AUC for each division: [0.54264918 0.53821095 0.54752191 0.56445739 0.54057542]
Average AUC: 0.5466829695688118
